# Deep Structural Inferential Analysis

## Overview
Statistical hypothesis testing for the deep structural decomposition analyses from notebook 03.

## Hypothesis Domains
- **D1**: Component-level band effects (per-component Jaccard within > between)
- **D2**: Edge characterization (sharing profiles, affinity vs distance, containment asymmetry)
- **D3**: Layer-level band effects (sensitivity profile, universal fraction trend)
- **D4**: Head-level band effects (universality distribution, discriminative heads)
- **D5**: Graph-theoretic band effects (diameter, clustering, hubs)
- **D6**: Input/output band effects
- **D7**: Variance decomposition of deep metrics
- **D8**: Hypothesis-specific tests (H5, H6, H7, H9)

All tests corrected with BH-FDR at α=0.05.

---
## 0. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from scipy import stats as sp_stats

sys.path.insert(0, str(Path.cwd()))

from utils import (
    MODELS,
    BANDS,
    FREQUENCY_BANDS,
    DRAWS,
    MODEL_INFO,
    BAND_NAMES,
    BAND_COLORS,
    MODEL_COLORS,
    FREQUENCY_RANK,
    ANALYSIS_DIR,
    VIZ_DIR,
    get_output_dirs,
    load_extracted_data,
    compute_component_jaccard,
    compute_band_affinity_summary,
    setup_plotting,
    save_figure,
)
from utils.stats import (
    TestAccumulator,
    cohens_d,
    rank_biserial,
    eta_squared,
    safe_kruskal,
    safe_mannwhitneyu,
    safe_spearmanr,
    jonckheere_terpstra,
    interpret_d,
    interpret_r,
    interpret_eta2,
)

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()

# Load circuit data
circuits, df = load_extracted_data()
print(f"Loaded {len(circuits)} circuits")

Loaded 75 circuits (0 failed)
Loaded 75 circuits


In [2]:
# Load deep analysis CSVs from notebook 03
df_comp_jaccard = pd.read_csv(ANALYSIS_DIR / "deep_component_jaccard.csv")
df_sharing_by_comp = pd.read_csv(ANALYSIS_DIR / "deep_sharing_by_component.csv")
df_sharing_profiles = pd.read_csv(ANALYSIS_DIR / "deep_sharing_profiles.csv")
df_band_affinity = pd.read_csv(ANALYSIS_DIR / "deep_band_affinity.csv")
df_band_signatures = pd.read_csv(ANALYSIS_DIR / "deep_band_signatures.csv")
df_containment = pd.read_csv(ANALYSIS_DIR / "deep_directed_containment.csv")
df_layer_sensitivity = pd.read_csv(ANALYSIS_DIR / "deep_layer_sensitivity.csv")
df_layer_universal = pd.read_csv(ANALYSIS_DIR / "deep_layer_universal_fraction.csv")
df_head_presence = pd.read_csv(ANALYSIS_DIR / "deep_head_universality.csv")
df_head_entropy = pd.read_csv(ANALYSIS_DIR / "deep_head_entropy.csv")
df_graph = pd.read_csv(ANALYSIS_DIR / "deep_graph_metrics.csv")
df_hub_nodes = pd.read_csv(ANALYSIS_DIR / "deep_hub_nodes.csv")
df_wiring = pd.read_csv(ANALYSIS_DIR / "deep_component_wiring.csv")

# S-G3/G5 gap analysis CSVs (generated by NB03 section 7)
_sg3_files = {
    "deep_stability_vs_sharing.csv": "df_stab_vs_sharing",
    "deep_edge_sharing_raw.csv": "df_edge_sharing_raw",
}
for fname, varname in _sg3_files.items():
    fpath = ANALYSIS_DIR / fname
    if fpath.exists():
        globals()[varname] = pd.read_csv(fpath)
        print(f"  Loaded {fname}")
    else:
        globals()[varname] = pd.DataFrame()
        print(f"  WARNING: {fname} not found (run NB03 section 7 first)")

print("All deep analysis CSVs loaded successfully.")

# Initialize test accumulator
acc = TestAccumulator()

  Loaded deep_stability_vs_sharing.csv
  Loaded deep_edge_sharing_raw.csv
All deep analysis CSVs loaded successfully.


---
## D1: Component-Level Band Effects

In [3]:
# D1.1: Per-component within > between Jaccard (one-sided Mann-Whitney)
component_types = ["attn", "mlp", "resid"]

for model in MODELS:
    for comp in component_types:
        result = compute_component_jaccard(circuits, model, comp)
        within = result["within"]
        between = result["between"]

        if len(within) >= 2 and len(between) >= 2:
            stat, p = sp_stats.mannwhitneyu(within, between, alternative="greater")
            es = cohens_d(within, between)

            acc.add_test(
                domain="D1_Component_Jaccard",
                hypothesis=f"{comp} within > between Jaccard",
                model=model,
                test_name="Mann-Whitney U (one-sided)",
                comparison="within_vs_between",
                statistic=stat,
                p_value=p,
                effect_size=es,
                effect_type="cohens_d",
                n1=len(within),
                n2=len(between),
            )

print(f"D1: {len(acc.tests)} tests so far")

D1: 15 tests so far


In [4]:
# D1.2: Component wiring differences by band (Kruskal-Wallis)
frac_cols = [c for c in df_wiring.columns if c.endswith("_frac")]

for model in MODELS:
    sub = df_wiring[df_wiring["model"] == model]
    for col in frac_cols:
        groups = [
            sub[sub["band"] == b][col].values
            for b in BANDS
            if len(sub[sub["band"] == b]) > 0
        ]
        if len(groups) >= 2 and all(len(g) >= 2 for g in groups):
            stat, p = sp_stats.kruskal(*groups)
            n = sum(len(g) for g in groups)
            es = (stat - len(groups) + 1) / (n - len(groups))

            acc.add_test(
                domain="D1_Component_Wiring",
                hypothesis=f"{col} differs by band",
                model=model,
                test_name="Kruskal-Wallis",
                comparison="omnibus_5bands",
                statistic=stat,
                p_value=p,
                effect_size=es,
                effect_type="eta_squared",
                n1=n,
                n2=len(groups),
            )

print(f"D1 total: {len(acc.tests)} tests")

<TMPDIR>/env/lib/python3.12/site-packages/scipy/stats/_stats_py.py:8492: RuntimeWarning: invalid value encountered in scalar divide
  h /= ties


D1 total: 60 tests


---
## D2: Edge Characterization

In [5]:
# D2.1: Band affinity correlation with frequency distance (Spearman, tests H7)
for model in MODELS:
    sub = df_band_affinity[
        (df_band_affinity["model"] == model) & df_band_affinity["freq_distance"].notna()
    ]
    if len(sub) >= 3:
        rho, p = sp_stats.spearmanr(sub["freq_distance"], sub["affinity"])
        acc.add_test(
            domain="D2_Edge_Characterization",
            hypothesis="affinity decreases with freq distance (H7)",
            model=model,
            test_name="Spearman",
            comparison="affinity_vs_distance",
            statistic=rho,
            p_value=p,
            effect_size=rho,
            effect_type="spearman_rho",
            n1=len(sub),
        )

# D2.2: Directed containment asymmetry (H5 subset hypothesis)
# Test: containment(low-freq source -> high-freq target) > containment(high-freq source -> low-freq target)
#
# NOTE: H5 containment asymmetry is model-dependent due to edge count ordering.
# For pythia-70m, high-freq bands have MORE edges than low-freq (e.g., low=389
# vs very_high=419), which inverts the expected containment direction;
# larger circuits mechanically contain more of smaller circuits' edges.
# For 160m/410m/1b, low-freq bands have more edges, supporting H5.
# This is why H5 is NOT significant for 70m but IS significant for 160m/410m/1b.
from utils.constants import LOW_FREQ_BANDS, HIGH_FREQ_BANDS

for model in MODELS:
    sub = df_containment[df_containment["model"] == model]

    # LF source, HF target
    lf_to_hf = sub[
        sub["source_band"].isin(LOW_FREQ_BANDS)
        & sub["target_band"].isin(HIGH_FREQ_BANDS)
    ]["containment"].values

    # HF source, LF target
    hf_to_lf = sub[
        sub["source_band"].isin(HIGH_FREQ_BANDS)
        & sub["target_band"].isin(LOW_FREQ_BANDS)
    ]["containment"].values

    if len(lf_to_hf) >= 2 and len(hf_to_lf) >= 2:
        stat, p = sp_stats.mannwhitneyu(lf_to_hf, hf_to_lf, alternative="greater")
        es = cohens_d(lf_to_hf, hf_to_lf)

        # Compute mean edge counts for context
        lf_sizes = sub[sub["source_band"].isin(LOW_FREQ_BANDS)]["source_size"].mean()
        hf_sizes = sub[sub["source_band"].isin(HIGH_FREQ_BANDS)]["source_size"].mean()
        size_note = f"mean_LF_edges={lf_sizes:.0f}, mean_HF_edges={hf_sizes:.0f}"

        acc.add_test(
            domain="D2_Edge_Characterization",
            hypothesis="LF contains HF edges > HF contains LF edges (H5)",
            model=model,
            test_name="Mann-Whitney U (one-sided)",
            comparison="containment_asymmetry",
            statistic=stat,
            p_value=p,
            effect_size=es,
            effect_type="cohens_d",
            n1=len(lf_to_hf),
            n2=len(hf_to_lf),
            edge_count_note=size_note,
        )

print(f"D2: {len(acc.tests)} tests")

D2: 70 tests


---
## D3: Layer-Level Band Effects

In [6]:
# D3.1: Layer sensitivity - identify which model shows significant layer variation
# Spearman: does sensitivity correlate with layer depth?
for model in MODELS:
    sub = df_layer_sensitivity[df_layer_sensitivity["model"] == model]
    if len(sub) >= 3:
        rho, p = sp_stats.spearmanr(sub["layer"], sub["mean_jaccard"])
        acc.add_test(
            domain="D3_Layer_Effects",
            hypothesis="layer sensitivity correlates with depth",
            model=model,
            test_name="Spearman",
            comparison="sensitivity_vs_depth",
            statistic=rho,
            p_value=p,
            effect_size=rho,
            effect_type="spearman_rho",
            n1=len(sub),
        )

# D3.2: Per-layer universal fraction trend across layers
for model in MODELS:
    sub = df_layer_universal[df_layer_universal["model"] == model]
    avg = sub.groupby("layer")["universal_fraction"].mean().reset_index()
    if len(avg) >= 3:
        rho, p = sp_stats.spearmanr(avg["layer"], avg["universal_fraction"])
        acc.add_test(
            domain="D3_Layer_Effects",
            hypothesis="universal fraction trends with layer depth",
            model=model,
            test_name="Spearman",
            comparison="universal_frac_vs_depth",
            statistic=rho,
            p_value=p,
            effect_size=rho,
            effect_type="spearman_rho",
            n1=len(avg),
        )

print(f"D3: {len(acc.tests)} tests")

D3: 80 tests


---
## D4: Head-Level Band Effects

In [7]:
# D4.1: Head universality - do heads vary significantly in their universality scores?
for model in MODELS:
    # Mean head entropy by band
    ent_sub = df_head_entropy[
        (df_head_entropy["model"] == model) & (df_head_entropy["total_edges"] > 0)
    ]
    if not ent_sub.empty:
        # Average entropy across heads: do some models have more discriminative heads?
        mean_ent = ent_sub.groupby("draw")["normalized_entropy"].mean().values
        overall_mean = mean_ent.mean()
        acc.add_test(
            domain="D4_Head_Effects",
            hypothesis="heads show band discrimination (entropy < 1)",
            model=model,
            test_name="Descriptive",
            comparison="mean_normalized_entropy",
            statistic=overall_mean,
            p_value=np.nan,
            effect_size=1.0 - overall_mean,
            effect_type="discrimination_score",
            n1=len(ent_sub),
        )

print(f"D4: {len(acc.tests)} tests")

D4: 85 tests


---
## D5: Graph-Theoretic Band Effects

In [8]:
# D5: Test graph metrics by band (Kruskal-Wallis per model)
graph_metrics = [
    "diameter",
    "clustering_coefficient",
    "density",
    "avg_path_length",
    "n_weakly_connected",
]

for model in MODELS:
    sub = df_graph[df_graph["model"] == model]
    for metric in graph_metrics:
        groups = [
            sub[sub["band"] == b][metric].dropna().values
            for b in BANDS
            if len(sub[sub["band"] == b]) > 0
        ]
        groups = [g for g in groups if len(g) >= 2]

        if len(groups) >= 2:
            stat, p = sp_stats.kruskal(*groups)
            n = sum(len(g) for g in groups)
            es = (stat - len(groups) + 1) / (n - len(groups))

            acc.add_test(
                domain="D5_Graph_Effects",
                hypothesis=f"{metric} differs by band",
                model=model,
                test_name="Kruskal-Wallis",
                comparison="omnibus_5bands",
                statistic=stat,
                p_value=p,
                effect_size=es,
                effect_type="eta_squared",
                n1=n,
                n2=len(groups),
            )

# D5.2: Hub universality test
for model in MODELS:
    sub = df_hub_nodes[
        (df_hub_nodes["model"] == model)
        & (df_hub_nodes["centrality_type"] == "betweenness")
    ]
    if not sub.empty:
        n_universal = sub["is_universal_hub"].sum()
        n_total = len(sub)
        acc.add_test(
            domain="D5_Graph_Effects",
            hypothesis="betweenness hubs are universal",
            model=model,
            test_name="Descriptive",
            comparison="hub_universality_rate",
            statistic=n_universal / n_total if n_total > 0 else 0,
            p_value=np.nan,
            effect_size=n_universal / n_total if n_total > 0 else 0,
            effect_type="proportion",
            n1=n_total,
        )

print(f"D5: {len(acc.tests)} tests")

D5: 115 tests


<TMPDIR>/env/lib/python3.12/site-packages/scipy/stats/_stats_py.py:8492: RuntimeWarning: invalid value encountered in scalar divide
  h /= ties


---
## D6: Input/Output Band Effects

In [9]:
# D6: Test if input/output edge distributions differ by band
df_input = pd.read_csv(ANALYSIS_DIR / "deep_input_edges.csv")
df_output = pd.read_csv(ANALYSIS_DIR / "deep_output_edges.csv")

# For each model: count input edges per bandxdraw, test Kruskal-Wallis
for model in MODELS:
    # Input edge count by band
    sub = df_input[df_input["model"] == model]
    counts = sub.groupby(["band", "draw"]).size().reset_index(name="n_input")
    groups = [
        counts[counts["band"] == b]["n_input"].values
        for b in BANDS
        if len(counts[counts["band"] == b]) > 0
    ]
    groups = [g for g in groups if len(g) >= 2]

    if len(groups) >= 2:
        stat, p = sp_stats.kruskal(*groups)
        n = sum(len(g) for g in groups)
        es = (stat - len(groups) + 1) / (n - len(groups))
        acc.add_test(
            domain="D6_IO_Effects",
            hypothesis="input edge count differs by band",
            model=model,
            test_name="Kruskal-Wallis",
            comparison="input_count_by_band",
            statistic=stat,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=n,
            n2=len(groups),
        )

    # Output edge count by band
    sub = df_output[df_output["model"] == model]
    counts = sub.groupby(["band", "draw"]).size().reset_index(name="n_output")
    groups = [
        counts[counts["band"] == b]["n_output"].values
        for b in BANDS
        if len(counts[counts["band"] == b]) > 0
    ]
    groups = [g for g in groups if len(g) >= 2]

    if len(groups) >= 2:
        stat, p = sp_stats.kruskal(*groups)
        n = sum(len(g) for g in groups)
        es = (stat - len(groups) + 1) / (n - len(groups))
        acc.add_test(
            domain="D6_IO_Effects",
            hypothesis="output edge count differs by band",
            model=model,
            test_name="Kruskal-Wallis",
            comparison="output_count_by_band",
            statistic=stat,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=n,
            n2=len(groups),
        )

print(f"D6: {len(acc.tests)} tests")

D6: 125 tests


---
## D7: Variance Decomposition

In [10]:
# D7: ANOVA on graph metrics: model, band, draw, modelxband
from itertools import product as iterproduct

for metric in ["diameter", "clustering_coefficient", "density"]:
    data = df_graph[[metric, "model", "band", "draw"]].dropna()
    if data.empty:
        continue

    # Simple two-way: model effect
    groups_model = [data[data["model"] == m][metric].values for m in MODELS]
    groups_model = [g for g in groups_model if len(g) >= 2]
    if len(groups_model) >= 2:
        stat, p = sp_stats.kruskal(*groups_model)
        n = sum(len(g) for g in groups_model)
        es = (stat - len(groups_model) + 1) / (n - len(groups_model))
        acc.add_test(
            domain="D7_Variance",
            hypothesis=f"{metric}: model effect",
            model="all",
            test_name="Kruskal-Wallis",
            comparison="model",
            statistic=stat,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=n,
        )

    # Band effect (within each model)
    for model in MODELS:
        sub = data[data["model"] == model]
        groups_band = [
            sub[sub["band"] == b][metric].values
            for b in BANDS
            if len(sub[sub["band"] == b]) > 0
        ]
        groups_band = [g for g in groups_band if len(g) >= 2]
        if len(groups_band) >= 2:
            stat, p = sp_stats.kruskal(*groups_band)
            n = sum(len(g) for g in groups_band)
            es = (stat - len(groups_band) + 1) / (n - len(groups_band))
            acc.add_test(
                domain="D7_Variance",
                hypothesis=f"{metric}: band effect",
                model=model,
                test_name="Kruskal-Wallis",
                comparison="band",
                statistic=stat,
                p_value=p,
                effect_size=es,
                effect_type="eta_squared",
                n1=n,
            )

print(f"D7: {len(acc.tests)} tests")

D7: 143 tests


---
## D8: Hypothesis-Specific Tests

In [11]:
# D8.1: H6: MLP edges have lower within-between Jaccard gap than attention edges?
# (i.e., MLP carries MORE variation)
for model in MODELS:
    sub = df_comp_jaccard[df_comp_jaccard["model"] == model]
    attn_gap = sub[sub["component"] == "attn"]["gap"].values
    mlp_gap = sub[sub["component"] == "mlp"]["gap"].values

    # Report which component has the larger gap
    if len(attn_gap) > 0 and len(mlp_gap) > 0:
        acc.add_test(
            domain="D8_Hypothesis_Specific",
            hypothesis="H6: MLP Jaccard gap vs attn Jaccard gap",
            model=model,
            test_name="Descriptive",
            comparison="attn_gap_vs_mlp_gap",
            statistic=mlp_gap[0] - attn_gap[0],
            p_value=np.nan,
            effect_size=mlp_gap[0] - attn_gap[0],
            effect_type="gap_difference",
            n1=1,
        )

# D8.2: H7: Overall affinity vs frequency distance
sub = df_band_affinity[df_band_affinity["freq_distance"].notna()]
if len(sub) >= 5:
    rho, p = sp_stats.spearmanr(sub["freq_distance"], sub["affinity"])
    acc.add_test(
        domain="D8_Hypothesis_Specific",
        hypothesis="H7: overall affinity vs freq distance",
        model="all",
        test_name="Spearman",
        comparison="affinity_vs_distance_all",
        statistic=rho,
        p_value=p,
        effect_size=rho,
        effect_type="spearman_rho",
        n1=len(sub),
    )

# D8.3: H9: Does layer sensitivity show significant variation? (range test)
for model in MODELS:
    sub = df_layer_sensitivity[df_layer_sensitivity["model"] == model]
    if len(sub) >= 3:
        sensitivity_range = sub["mean_jaccard"].max() - sub["mean_jaccard"].min()
        cv = (
            sub["mean_jaccard"].std() / sub["mean_jaccard"].mean()
            if sub["mean_jaccard"].mean() > 0
            else 0
        )
        acc.add_test(
            domain="D8_Hypothesis_Specific",
            hypothesis="H9: layer sensitivity shows variation (CV)",
            model=model,
            test_name="Descriptive",
            comparison="sensitivity_CV",
            statistic=cv,
            p_value=np.nan,
            effect_size=sensitivity_range,
            effect_type="range",
            n1=len(sub),
        )

print(f"D8: {len(acc.tests)} tests")

D8: 154 tests


---
## D9: Draw Stability & Cross-Analyses

- **DS1**: Chi-squared test -- is draw stability associated with sharing level? (per model, 4 tests)
- **DS2**: Mann-Whitney -- is the draw-stable fraction higher for universal edges than band-specific? (per model, 4 tests)
- **SK1**: Mann-Whitney -- is layer_distance different for sharing=1 vs sharing=5 edges? (per model, 4 tests)
- **SK2**: Spearman -- sharing_level vs layer_distance correlation (per model, 4 tests)

In [12]:
# D9: Draw Stability Tests (S-G3)
from scipy.stats import chi2_contingency

if not df_stab_vs_sharing.empty:
    # DS1: Chi-squared -- draw stability associated with sharing level?
    for model in MODELS:
        sub = df_stab_vs_sharing[
            (df_stab_vs_sharing["model"] == model)
            & df_stab_vs_sharing["sharing_level"].notna()
        ]
        if sub.empty:
            continue
        ct = pd.crosstab(sub["n_draws"], sub["sharing_level"])
        if ct.shape[0] >= 2 and ct.shape[1] >= 2:
            chi2, p, dof, expected = chi2_contingency(ct)
            # Cramer's V as effect size
            n = ct.values.sum()
            k = min(ct.shape) - 1
            cramers_v = np.sqrt(chi2 / (n * k)) if n * k > 0 else 0
            acc.add_test(
                domain="D9_DrawStability",
                hypothesis="DS1: draw stability associated with sharing level",
                model=model,
                test_name="Chi-squared",
                comparison="stability_x_sharing",
                statistic=chi2,
                p_value=p,
                effect_size=cramers_v,
                effect_type="cramers_v",
                n1=n,
                dof=dof,
            )

    # DS2: Mann-Whitney -- universal edges more draw-stable than band-specific?
    for model in MODELS:
        sub = df_stab_vs_sharing[
            (df_stab_vs_sharing["model"] == model)
            & df_stab_vs_sharing["sharing_level"].notna()
        ]
        universal = sub[sub["sharing_level"] == 5]["n_draws"].values
        band_specific = sub[sub["sharing_level"] == 1]["n_draws"].values

        if len(universal) >= 2 and len(band_specific) >= 2:
            stat, p = safe_mannwhitneyu(universal, band_specific, alternative="greater")
            es = rank_biserial(universal, band_specific)
            acc.add_test(
                domain="D9_DrawStability",
                hypothesis="DS2: universal edges more draw-stable than band-specific",
                model=model,
                test_name="Mann-Whitney U (one-sided)",
                comparison="universal_vs_band_specific_stability",
                statistic=stat,
                p_value=p,
                effect_size=es,
                effect_type="rank_biserial",
                n1=len(universal),
                n2=len(band_specific),
            )

    print(f"D9 draw stability tests added: {len(acc.tests)} tests so far")
else:
    print("WARNING: df_stab_vs_sharing is empty -- skipping DS1/DS2 tests")

D9 draw stability tests added: 164 tests so far


In [13]:
# D9: Skip Distance Tests (S-G5)
if not df_edge_sharing_raw.empty:
    # SK1: Mann-Whitney -- layer_distance for sharing=1 vs sharing=5
    for model in MODELS:
        sub = df_edge_sharing_raw[
            (df_edge_sharing_raw["model"] == model)
            & (df_edge_sharing_raw["draw"] == "draw_1")
        ]
        universal_dist = sub[sub["sharing_level"] == 5]["layer_distance"].values
        specific_dist = sub[sub["sharing_level"] == 1]["layer_distance"].values

        if len(universal_dist) >= 2 and len(specific_dist) >= 2:
            stat, p = safe_mannwhitneyu(
                universal_dist, specific_dist, alternative="two-sided"
            )
            es = rank_biserial(universal_dist, specific_dist)
            acc.add_test(
                domain="D9_SkipDistance",
                hypothesis="SK1: layer_distance differs for universal vs band-specific edges",
                model=model,
                test_name="Mann-Whitney U (two-sided)",
                comparison="universal_vs_specific_layer_distance",
                statistic=stat,
                p_value=p,
                effect_size=es,
                effect_type="rank_biserial",
                n1=len(universal_dist),
                n2=len(specific_dist),
            )

    # SK2: Spearman -- sharing_level vs layer_distance
    for model in MODELS:
        sub = df_edge_sharing_raw[
            (df_edge_sharing_raw["model"] == model)
            & (df_edge_sharing_raw["draw"] == "draw_1")
        ]
        if len(sub) >= 5:
            rho, p = safe_spearmanr(
                sub["sharing_level"].values, sub["layer_distance"].values
            )
            acc.add_test(
                domain="D9_SkipDistance",
                hypothesis="SK2: sharing_level correlates with layer_distance",
                model=model,
                test_name="Spearman",
                comparison="sharing_vs_layer_distance",
                statistic=rho,
                p_value=p,
                effect_size=rho,
                effect_type="spearman_rho",
                n1=len(sub),
            )

    print(f"D9 skip distance tests added: {len(acc.tests)} total tests")
else:
    print("WARNING: df_edge_sharing_raw is empty -- skipping SK1/SK2 tests")

D9 skip distance tests added: 174 total tests


---
## Results Summary

In [14]:
# Apply FDR correction and export
df_tests = acc.to_dataframe()
df_tests = acc.apply_fdr_correction(df_tests)

df_tests.to_csv(ANALYSIS_DIR / "deep_all_hypothesis_tests.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_all_hypothesis_tests.csv'}")
print(f"\nTotal tests: {len(df_tests)}")

# Count significant by domain
testable = df_tests[df_tests["p_value"].notna()]
sig_uncorrected = (
    testable["significant_uncorrected"].sum()
    if "significant_uncorrected" in testable.columns
    else 0
)
sig_bh = testable["significant_bh"].sum() if "significant_bh" in testable.columns else 0
print(f"Significant (uncorrected): {sig_uncorrected}/{len(testable)}")
print(f"Significant (BH-FDR): {sig_bh}/{len(testable)}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_all_hypothesis_tests.csv

Total tests: 174


Significant (uncorrected): 64/147
Significant (BH-FDR): 37/147


In [15]:
# Summary by domain
testable = df_tests[df_tests["p_value"].notna()].copy()
domain_summary = (
    testable.groupby("domain")
    .agg(
        n_tests=("p_value", "count"),
        n_sig_uncorrected=("significant_uncorrected", "sum")
        if "significant_uncorrected" in testable.columns
        else ("p_value", lambda x: 0),
        n_sig_bh=("significant_bh", "sum")
        if "significant_bh" in testable.columns
        else ("p_value", lambda x: 0),
        mean_effect=("effect_size", lambda x: x.dropna().mean()),
    )
    .reset_index()
)

print("\n=== Results by Domain ===")
print(domain_summary.to_string(index=False))


=== Results by Domain ===
                  domain  n_tests  n_sig_uncorrected  n_sig_bh  mean_effect
    D1_Component_Jaccard       15                 14        13     0.793048
     D1_Component_Wiring       43                 17         0     0.369456
D2_Edge_Characterization       10                  5         5     0.634995
        D3_Layer_Effects       10                  3         2    -0.092418
        D5_Graph_Effects       20                  3         0     0.178031
           D6_IO_Effects       10                  1         0     0.252260
             D7_Variance       18                  5         3     0.249889
  D8_Hypothesis_Specific        1                  1         0    -0.443293
        D9_DrawStability       10                 10        10     0.728033
         D9_SkipDistance       10                  5         4    -0.018770


In [16]:
# Visualization: Significance rates by domain
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Test counts by domain
ax = axes[0]
domains = domain_summary["domain"].values
x = np.arange(len(domains))
ax.barh(x, domain_summary["n_tests"], color="lightgray", label="Total")
if "n_sig_bh" in domain_summary.columns:
    ax.barh(x, domain_summary["n_sig_bh"], color="#d62728", label="Significant (BH)")
ax.set_yticks(x)
ax.set_yticklabels(domains, fontsize=9)
ax.set_xlabel("Number of Tests")
ax.set_title("Significance by Domain")
ax.legend()
ax.invert_yaxis()

# Panel 2: P-value distribution
ax = axes[1]
p_vals = testable["p_value"].dropna()
ax.hist(p_vals, bins=20, color="steelblue", alpha=0.7, edgecolor="white")
ax.axvline(x=0.05, color="red", linestyle="--", label="α=0.05")
ax.set_xlabel("P-value")
ax.set_ylabel("Count")
ax.set_title("P-value Distribution (all tests)")
ax.legend()

fig.tight_layout()
save_figure(fig, "deep_17_significance_summary.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_17_significance_summary.png


In [17]:
# Display significant results
if "significant_bh" in df_tests.columns:
    sig = df_tests[df_tests["significant_bh"] == True]
else:
    sig = df_tests[df_tests["p_value"] < 0.05]

print(f"\n=== Significant Results (BH-FDR corrected) ===")
if not sig.empty:
    display_cols = [
        "domain",
        "hypothesis",
        "model",
        "test",
        "statistic",
        "p_value",
        "effect_size",
        "effect_type",
        "interpretation",
    ]
    display_cols = [c for c in display_cols if c in sig.columns]
    print(sig[display_cols].to_string(index=False))
else:
    print("No significant results after BH-FDR correction.")


=== Significant Results (BH-FDR corrected) ===
                  domain                                                       hypothesis       model                       test     statistic       p_value  effect_size   effect_type interpretation
    D1_Component_Jaccard                                    attn within > between Jaccard  pythia-70m Mann-Whitney U (one-sided)  2.068000e+03  6.849819e-06     0.990567      cohens_d          large
    D1_Component_Jaccard                                     mlp within > between Jaccard  pythia-70m Mann-Whitney U (one-sided)  1.831000e+03  1.789633e-03     0.744989      cohens_d         medium
    D1_Component_Jaccard                                   resid within > between Jaccard  pythia-70m Mann-Whitney U (one-sided)  1.917000e+03  2.331209e-04     0.769170      cohens_d         medium
    D1_Component_Jaccard                                    attn within > between Jaccard pythia-160m Mann-Whitney U (one-sided)  2.288000e+03  6.657751e-09